# Constrained Decoding for Triton GPU Kernels
### *Does grammar-guided generation + few-shot patterns help a small LLM write valid Triton kernels?*

**Research question.** Do XGrammar-constrained decoding and a few-shot pattern library improve `call@1` and `exe@1` for a small open-source coder model (Qwen2.5-Coder-7B) on the **TritonBench-T** benchmark — and how does the small model compare against two frontier APIs (Claude, DeepSeek)?

We compare **5 regimes** on the same 50 randomly-sampled TritonBench-T operators:

| Regime | Model | Decoding | Few-shot |
|---|---|---|---|
| `qwen_free` | Qwen2.5-Coder-7B (local, int4) | unconstrained | — |
| `qwen_constrained` | Qwen2.5-Coder-7B | XGrammar (Triton EBNF) + Python-keyword mask | — |
| `qwen_patterns` | Qwen2.5-Coder-7B | XGrammar + keyword mask | one validated Triton pattern, picked by keyword heuristic |
| `claude` | Claude Sonnet 4.6 (API) | unconstrained | — |
| `deepseek` | DeepSeek-V3 (API) | unconstrained | — |

**Metrics.** TritonBench-T's official 3-phase evaluator:

- `call@1` — kernel compiles and runs without crashing
- `exe@1` — output matches the PyTorch reference numerically
- `speedup` — wall-clock vs the PyTorch reference (only for kernels that pass `exe@1`)

### Index

1. Environment + GPU check
2. Pick the small model
3. EBNF grammar consumed by XGrammar
4. Triton pattern library + pattern-picker heuristic
5. Shared helpers (SYSTEM_PROMPT, output cleaner, forbidden-tokens processor)
6. TritonBench-T setup
7. Generation for the 5 regimes
8. Phase 1 + Phase 2 evaluation (`call@1`, `exe@1`)
9. Phase 3 evaluation (speedup)
10. Final comparison table

---

> Runtime: A100 or T4 GPU on Colab. With a 50-operator subset: **~2 hours**. Full 166: **~5–6 hours**.


## 1. Environment setup

In [ ]:
# Verifica si hay GPU disponible y muestra sus características

import subprocess, sys, os
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]
    ).decode().strip()
    print("GPU detected:", out)
except Exception:
    print("No GPU found — switch the Colab runtime to GPU (Runtime → Change runtime type → GPU).")
    raise SystemExit(1)

import torch
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA must be available"


GPU detected: NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB, 580.82.07
torch  : 2.11.0+cu128
cuda   : 12.8


### Install dependencies

We need:
- `transformers` + `accelerate` (load the small LM)
- `xgrammar` (constrained decoding)
- `triton` (so we can actually compile/run generated kernels)

Colab usually has the first two and Triton ships with a recent PyTorch. `xgrammar` is the new one.


In [ ]:
import subprocess
import sys
import importlib.metadata

# Función auxiliar para instalar paquetes de manera silenciosa
def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# xgrammar requires a recent build; pin one we have tested with transformers >=4.45
try:
    import xgrammar  # noqa: F401
except ImportError:
    _pip("xgrammar>=0.1.18")

try:
    import transformers  # noqa: F401
except ImportError:
    _pip("transformers>=4.45.0", "accelerate>=0.30.0")

# bitsandbytes para cuantización int4 (necesaria para cargar Qwen 7B en T4)
try:
    import bitsandbytes  # noqa: F401
    if importlib.metadata.version("bitsandbytes") < "0.46.1":
        raise ImportError("bitsandbytes too old")
except ImportError:
    _pip("bitsandbytes>=0.46.1", "accelerate>=0.30.0")

import xgrammar as xgr
import transformers, triton

print("transformers :", transformers.__version__)
print("xgrammar     :", importlib.metadata.version("xgrammar"))
print("triton       :", triton.__version__)
print("bitsandbytes :", importlib.metadata.version("bitsandbytes"))

transformers : 5.0.0
xgrammar     : 0.2.1
triton       : 3.6.0
bitsandbytes : 0.49.2


## 2. Pick a Small Language Model

Wee needo to choose an **SLM**. We want something that fits on a free T4 (~15 GB VRAM) and that has at least seen Triton during training. Qwen/Qwen2.5-Coder-7B-Instruct is the one we chose.


In [ ]:
# Cargar el tokenizador y modelo de lenguaje (7B parámetros, cuantizado a int4)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"   # 7B en int4 ≈ 5 GB VRAM; cabe en T4

# Configuración de cuantizacion int4 (NF4) — reduce VRAM de ~14 GB a ~5 GB
# con pérdida de calidad minima (<2 pp en benchmarks de código)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",   # auto = pone capas en GPU, el resto en CPU si hace falta
)
model.eval()
print(f"Loaded {MODEL_ID}")
print(f"Vocab size  : {tokenizer.vocab_size}")
print(f"VRAM in use : {torch.cuda.memory_allocated()/1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-Coder-7B-Instruct
Vocab size  : 151643
VRAM in use : 5.59 GB


## 3. EBNF Grammar for XGrammar

XGrammar accepts a context-free grammar in **EBNF** (Extended Backus-Naur Form). At each decoding step it computes a **token mask** over the model's vocabulary: tokens that would extend the current prefix into a string outside the language get `-inf` logits, so they cannot be sampled.

### Design decision: keep the grammar shallow

A *complete* Python grammar would be huge and would let the model produce Python that isn't Triton. We instead encode **the shape of a Triton kernel translation**:

```
output ::= imports kernel_decl wrapper_decl
imports ::= "import torch\nimport triton\nimport triton.language as tl\n"
kernel_decl ::= "@triton.jit\n" func_def
wrapper_decl ::= func_def
func_def ::= "def" IDENT "(" params "):" block
...
```

This is *intentionally narrower* than the lexical spec — it's a working subset that biases the model towards the canonical kernel skeleton (`pid`, `offsets`, `mask`, `load`, `store`). Free generation has none of this scaffolding.

### What the grammar guarantees vs. doesn't

- The output **starts with** the right imports.
- The output **contains** an `@triton.jit`-decorated function.
- Every identifier comes from the allowed alphabet.
- No reserved-rejected keywords (`class`, `lambda`, `yield`, …) can appear.

It does **not** guarantee that masks are placed correctly, that `axis=0` is passed to `tl.sum`, or that the math is right. Those are semantic concerns — that's what `exe@1` will catch.


In [ ]:
# La gramática EBNF que XGrammar usará para validar tokens en cada paso.
# Describe la estructura sintáctica válida de kernels Triton.
# XGrammar la compila en un autómata pushdown para enmascarar tokens inválidos
# durante el decoding.

TRITON_EBNF = r"""
root        ::= header kernel_fn+ wrapper_fn

# ---------------- header ----------------
# Imports obligatorios primero; después permitimos imports extra opcionales
# (por ejemplo `import math`, `from triton.language import constexpr`) para
# que el modelo no se trabe si decide importar utilitarios adicionales.
header      ::= required_imports extra_import* "\n"
required_imports ::= "import torch\nimport triton\nimport triton.language as tl\n"
extra_import ::= "import " dotted_name "\n"
              |  "from " dotted_name " import " ident ("," ws ident)* "\n"
dotted_name ::= ident ("." ident)*

# ---------------- kernel ----------------
kernel_fn   ::= "@triton.jit\ndef " ident "(" params ")" return_ann? ":\n" kernel_body
return_ann  ::= " -> " attr_chain

params      ::= param ("," ws param)*
param       ::= ident type_ann? default_assign?
type_ann    ::= ":" ws attr_chain
default_assign ::= ws "=" ws default_val

# Cuerpo del kernel: una o más líneas, permite líneas en blanco intercaladas
# para que el modelo pueda separar secciones (offsets / load / compute / store).
kernel_body ::= kernel_line+
kernel_line ::= indent stmt "\n" | "\n"

# Statements permitidos dentro del kernel:
#  - assign      : x = expr
#  - aug_assign  : x += expr (común en acumuladores)
#  - return      : return expr
#  - expr_stmt   : llamada con efecto colateral (tl.store, tl.atomic_add, etc.)
#  - for_stmt    : for k in range(...): loops de tiles (matmul, conv, attention)
#  - if_stmt     : control de flujo condicional dentro del kernel
stmt        ::= assign_stmt
              | aug_assign_stmt
              | return_stmt
              | expr_stmt
              | for_stmt
              | if_stmt

assign_stmt ::= lhs ws "=" ws expr
aug_assign_stmt ::= lhs ws aug_op ws expr
aug_op      ::= "+=" | "-=" | "*=" | "/=" | "%=" | "&=" | "|=" | "^="
lhs         ::= attr_chain | subscript
return_stmt ::= "return" (ws expr)?
expr_stmt   ::= call

# Loops: `for k in range(0, N, BLOCK):` con cuerpo indentado dos niveles.
for_stmt    ::= "for " ident " in " expr ":\n" nested_body
if_stmt     ::= "if " expr ":\n" nested_body (indent "else:\n" nested_body)?
nested_body ::= nested_line+
nested_line ::= indent indent stmt "\n" | "\n"

# ---------------- expressions ----------------
# Limitadas pero suficientes para elementwise, reducciones, matmul-tipo.
expr        ::= unary_op? term (ws bin_op ws term)*
unary_op    ::= "-" | "+" | "~"
term        ::= call | subscript | attr_chain | number | "(" expr ")"

# Cadenas de atributos: x, x.numel, x.shape, data.device, etc.
# (sin string literals — no los necesitamos para kernels)
attr_chain  ::= ident ("." ident)*

# Subscript: x[i], x.shape[0], offsets[mask], x[None, :]
subscript   ::= attr_chain "[" expr "]"

call        ::= callee "(" args? ")"
callee      ::= "tl." tl_name | "triton." tr_name | attr_chain
args        ::= kw_or_pos ("," ws kw_or_pos)*
kw_or_pos   ::= (ident ws "=" ws)? expr

# Vocabulario de tl.*: ampliado para cubrir operadores reales de TritonBench-T.
# Incluye: program metadata, memoria, reducciones, funciones matemáticas,
# operaciones de shape, atomics y constructores.
tl_name     ::= "program_id" | "num_programs"
              | "arange" | "load" | "store"
              | "sum" | "max" | "min" | "mean" | "argmax" | "argmin"
              | "where" | "maximum" | "minimum" | "clamp"
              | "exp" | "exp2" | "log" | "log2" | "sqrt" | "rsqrt"
              | "sin" | "cos" | "tanh" | "sigmoid"
              | "abs" | "floor" | "ceil"
              | "atomic_add" | "atomic_max" | "atomic_min" | "atomic_cas"
              | "constexpr" | "rand" | "randn" | "zeros" | "full" | "ones"
              | "dot" | "trans" | "reshape" | "broadcast_to" | "expand_dims"
              | "cumsum" | "cumprod" | "flip"
              | "multiple_of" | "max_contiguous"

# Vocabulario de triton.*: utilidades del host.
tr_name     ::= "cdiv" | "next_power_of_2" | "jit"

# Operadores binarios: aritméticos, comparación, bitwise, shifts.
bin_op      ::= "+" | "-" | "*" | "**" | "/" | "//" | "%"
              | "<<" | ">>" | "^" | "&" | "|"
              | "<" | ">" | "<=" | ">=" | "==" | "!="
              | "and" | "or"

# ---------------- wrapper (python host code) ----------------
wrapper_fn  ::= "\ndef " ident "(" wparams ")" return_ann? ":\n" wrapper_body

# Wrapper params: permitir defaults estilo `alpha=1.0` para que firmas como
# `add(input, other, alpha=1.0)` sean alcanzables.
wparams     ::= wparam ("," ws wparam)*
wparam      ::= ident default_assign?
default_val ::= number | ident | "None" | "True" | "False"

wrapper_body::= wrapper_line+
wrapper_line::= indent wstmt "\n" | "\n"
wstmt       ::= assign_stmt | aug_assign_stmt | launch_stmt | return_stmt | expr_stmt | for_stmt | if_stmt
launch_stmt ::= attr_chain "[" expr "]" "(" args? ")"

# ---------------- léxico ----------------
# `ident` permite cualquier identificador léxico válido. El filtrado de
# Python keywords se hace FUERA de la gramática vía ForbiddenTokensProcessor
# — XGrammar no soporta negative lookahead, así que la defensa contra
# keywords es a nivel logits processor, no a nivel gramatical.
ident       ::= ident_start ident_rest*
ident_start ::= [A-Za-z_]
ident_rest  ::= [A-Za-z0-9_]

# Números: enteros, floats con o sin parte entera, notación científica.
number      ::= [0-9]+ ("." [0-9]*)? ([eE] [+-]? [0-9]+)?
              | "." [0-9]+ ([eE] [+-]? [0-9]+)?

ws          ::= " "*
indent      ::= "    "
"""

print("EBNF length:", len(TRITON_EBNF), "characters")
print("First 400 chars:\n", TRITON_EBNF[:400], "...")

EBNF length: 5493 characters
First 400 chars:
 
root        ::= header kernel_fn+ wrapper_fn

# ---------------- header ----------------
# Imports obligatorios primero; después permitimos imports extra opcionales
# (por ejemplo `import math`, `from triton.language import constexpr`) para
# que el modelo no se trabe si decide importar utilitarios adicionales.
header      ::= required_imports extra_import* "\n"
required_imports ::= "import torch ...


### 3.1 Compile the grammar with XGrammar

XGrammar needs (1) the grammar and (2) tokenizer info to pre-compute the per-state token masks. The compilation is a one-time cost.

In [ ]:
# Envolver tokenizador HuggingFace para que XGrammar pueda acceder al vocabulario
import xgrammar as xgr

# Compilar la gramática EBNF en un autómata pushdown
# Esta compilación es cara (1-2 seg), pero se cachea en disco
# Step 1 — wrap the HF tokenizer so XGrammar can iterate the vocab
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=model.config.vocab_size)

# Step 2 — compile the grammar
# Procesar errores de compilación: si la gramática es inválida, reportar y salir
compiler = xgr.GrammarCompiler(tokenizer_info)
try:
    triton_grammar = compiler.compile_grammar(TRITON_EBNF)
    print("Triton EBNF compiled correctly")
except Exception as e:
    print("EBNF compile failed:", e)
    raise


Triton EBNF compiled correctly


## 4. Triton Patterns + Pattern Picker

We seed the `qwen_patterns` regime with a small library of validated Triton kernels (built from the official Triton tutorials). For each TritonBench-T item, a keyword heuristic picks the most relevant pattern and we inject it into the user prompt as a single few-shot example.

The hypothesis:

- **Grammar (XGrammar)** → enforces *syntactic* validity (token-level).
- **Patterns** → guides *semantic* validity (which Triton idioms to use for the given operation).
- **Combined** → higher `call@1` and `exe@1` without changing the model or hardware.

The pattern names (`example_kernel`, `arg_tensor`, `scalar_factor`, …) are deliberately abstract so the model cannot copy them verbatim — it has to adapt the structure to the wrapper signature the benchmark expects.


In [ ]:
# Triton Patterns: catálogo de kernels validados (de los tutoriales oficiales).
# IMPORTANTE: nombres deliberadamente ABSTRACTOS (example_kernel, example_op,
# arg_tensor, scalar_factor) — para que el modelo NO los copie literalmente y
# respete la firma que pide el item de TritonBench-T.

PATTERNS = {
    "scale_mask": """\
@triton.jit
def example_kernel(in_ptr, out_ptr, scalar_factor, n_elems, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elems
    x = tl.load(in_ptr + offsets, mask=mask, other=0.0)
    tl.store(out_ptr + offsets, x * scalar_factor, mask=mask)


def example_op(arg_tensor, scalar_factor=2.0):
    n_elems = arg_tensor.numel()
    out = torch.empty_like(arg_tensor)
    BLOCK = 256
    grid = (triton.cdiv(n_elems, BLOCK),)
    example_kernel[grid](arg_tensor, out, scalar_factor, n_elems, BLOCK_SIZE=BLOCK)
    return out""",

    "reduce_two_stage": """\
@triton.jit
def example_stage1(in_ptr, temp_ptr, n_elems, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < n_elems
    data = tl.load(in_ptr + offsets, mask=mask, other=0.0)
    tl.store(temp_ptr + pid, tl.sum(data, axis=0))


@triton.jit
def example_stage2(temp_ptr, result_ptr, num_blocks, BLOCK: tl.constexpr):
    offsets = tl.arange(0, BLOCK)
    mask = offsets < num_blocks
    temp = tl.load(temp_ptr + offsets, mask=mask, other=0.0)
    tl.store(result_ptr, tl.sum(temp, axis=0))


def example_op(arg_tensor):
    n_elems = arg_tensor.numel()
    BLOCK = 256
    n_blocks = triton.cdiv(n_elems, BLOCK)
    temp = torch.zeros(n_blocks, device=arg_tensor.device)
    result = torch.zeros(1, device=arg_tensor.device)
    example_stage1[(n_blocks,)](arg_tensor, temp, n_elems, BLOCK=BLOCK)
    example_stage2[(1,)](temp, result, n_blocks, BLOCK=triton.next_power_of_2(n_blocks))
    return result[0]""",

    "softmax_row_reduce": """\
@triton.jit
def example_kernel(in_ptr, out_ptr, n_cols, BLOCK_SIZE: tl.constexpr):
    row_idx = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    row = tl.load(in_ptr + row_idx * n_cols + cols, mask=mask, other=0.0)
    row_max = tl.max(row, axis=0)
    row_minus_max = row - row_max
    numerator = tl.exp(row_minus_max)
    denominator = tl.sum(numerator, axis=0)
    result = numerator / denominator
    tl.store(out_ptr + row_idx * n_cols + cols, result, mask=mask)


def example_op(arg_tensor):
    n_rows = arg_tensor.shape[0]
    n_cols = arg_tensor.shape[1]
    BLOCK = triton.next_power_of_2(n_cols)
    out = torch.empty_like(arg_tensor)
    example_kernel[(n_rows,)](arg_tensor, out, n_cols, BLOCK_SIZE=BLOCK)
    return out""",

    "layernorm_single_pass": """\
@triton.jit
def example_kernel(in_ptr, out_ptr, n_cols, eps, BLOCK_SIZE: tl.constexpr):
    row_idx = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    x = tl.load(in_ptr + row_idx * n_cols + cols, mask=mask, other=0.0)
    mean = tl.sum(x, axis=0) / n_cols
    x_centered = x - mean
    var = tl.sum(x_centered * x_centered, axis=0) / n_cols
    rstd = 1.0 / tl.sqrt(var + eps)
    y = x_centered * rstd
    tl.store(out_ptr + row_idx * n_cols + cols, y, mask=mask)


def example_op(arg_tensor, eps=1e-5):
    n_rows = arg_tensor.shape[0]
    n_cols = arg_tensor.shape[1]
    BLOCK = triton.next_power_of_2(n_cols)
    out = torch.empty_like(arg_tensor)
    example_kernel[(n_rows,)](arg_tensor, out, n_cols, eps, BLOCK_SIZE=BLOCK)
    return out""",

    "dropout_with_mask": """\
@triton.jit
def example_kernel(in_ptr, keep_ptr, out_ptr, scalar_p, n_elems, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elems
    x = tl.load(in_ptr + offsets, mask=mask, other=0.0)
    keep = tl.load(keep_ptr + offsets, mask=mask, other=0)
    y = tl.where(keep > 0, x / (1.0 - scalar_p), 0.0)
    tl.store(out_ptr + offsets, y, mask=mask)


def example_op(arg_tensor, keep_mask, scalar_p):
    n_elems = arg_tensor.numel()
    out = torch.empty_like(arg_tensor)
    BLOCK = 1024
    grid = (triton.cdiv(n_elems, BLOCK),)
    example_kernel[grid](arg_tensor, keep_mask, out, scalar_p, n_elems, BLOCK_SIZE=BLOCK)
    return out""",
}


def pick_pattern_for_instruction(instruction: str) -> str:
    """Heurística por keywords: para cada item de TritonBench-T elige el patrón más
    relevante de la librería. Orden importa — patrones más específicos primero."""
    s = instruction.lower()
    if any(k in s for k in ("layer_norm", "layernorm", "layer norm",
                            "rms_norm", "rmsnorm", "rms norm",
                            "batch_norm", "batchnorm", "batch norm",
                            "instance_norm", "group_norm", "normalize", "normalization")):
        return "layernorm_single_pass"
    if "softmax" in s or "logsoftmax" in s or "log_softmax" in s:
        return "softmax_row_reduce"
    if "dropout" in s:
        return "dropout_with_mask"
    if any(k in s for k in ("reduction", "reduce", "sum over", "mean over",
                            "variance", "std deviation", "all elements",
                            "across all", "argmax", "argmin",
                            " sum(", " mean(", " max(", " min(")):
        return "reduce_two_stage"
    return "scale_mask"


print(f"Patterns disponibles: {list(PATTERNS.keys())}")
print()
for sample in [
    "Compute element-wise ReLU activation on a 1-D tensor.",
    "Apply row-wise softmax on a 2-D tensor.",
    "Compute layer normalization over the last dimension.",
    "Reduce a tensor by summing across all elements.",
    "Apply dropout with a Bernoulli mask.",
]:
    print(f"  '{sample[:55]:55s}' -> {pick_pattern_for_instruction(sample)}")


Patterns disponibles: ['scale_mask', 'reduce_two_stage', 'softmax_row_reduce', 'layernorm_single_pass', 'dropout_with_mask']

  'Compute element-wise ReLU activation on a 1-D tensor.  ' -> scale_mask
  'Apply row-wise softmax on a 2-D tensor.                ' -> softmax_row_reduce
  'Compute layer normalization over the last dimension.   ' -> layernorm_single_pass
  'Reduce a tensor by summing across all elements.        ' -> reduce_two_stage
  'Apply dropout with a Bernoulli mask.                   ' -> dropout_with_mask


## 5. Shared decoding helpers

Three things every regime needs:

1. **`SYSTEM_PROMPT`** — the system instruction shown to Qwen, Claude, and DeepSeek. Same prompt across regimes so the only varying knob is the decoding strategy (and the optional few-shot pattern).
2. **`process_code()`** — strips markdown fences, chat-template artefacts, and trailing prose; iteratively truncates from the end until `ast.parse()` succeeds. Mirrors what TritonBench-T's official evaluator does internally.
3. **`ForbiddenTokensProcessor`** + **`FORBIDDEN_IDS`** — a `LogitsProcessor` that masks Python keywords (`class`, `try`, `with`, `lambda`, …) that don't belong in a Triton kernel body. Used together with the XGrammar processor for the `constrained` / `patterns` regimes.


In [ ]:
SYSTEM_PROMPT = """\
You are an expert at writing Triton GPU kernels in Python.

Given a natural-language specification of a tensor operation, write:
1. One or more `@triton.jit` kernel function(s) using `triton.language` aliased as `tl`.
2. A Python wrapper that allocates the output, computes the grid, and launches the kernel.

Hard requirements (the grader checks these):
- Output ONLY runnable Python code. No prose, no markdown fences, no commentary.
- Import `torch`, `triton`, and `triton.language as tl` at the top.
- Use the standard Triton skeleton: `pid = tl.program_id(0)`,
  `offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)`,
  `mask = offsets < n_elements`,
  `tl.load(..., mask=mask, other=0.0)`,
  `tl.store(..., mask=mask)`.
- Pass scalar args by value; pass tensor args directly (NOT via `.data_ptr()`).
- The grid must be a tuple — `grid = (triton.cdiv(n, BLOCK),)` — note the trailing comma.
- Keep the wrapper signature exactly as suggested by the specification (argument names + defaults).
- Mark `BLOCK_SIZE` as `tl.constexpr` in the kernel signature.

Common pitfalls to avoid:
- For row-wise reductions, use `BLOCK_SIZE = triton.next_power_of_2(n_cols)`.
- For global reductions, use a two-stage pattern with a temp tensor.
- Do NOT wrap the answer in a Python class. Control flow (`for`, `if`) is OK
  inside the kernel body when the operator needs it (matmul tiles, masked
  conditionals), but keep it minimal.
"""


# ── 2. process_code(): mismo extractor que TritonBench-T ───────────────────
import ast


def _extract_imports_and_funcs(tree):
    imports, funcs = [], []
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            imports.append(ast.unparse(node))
        elif isinstance(node, ast.FunctionDef):
            funcs.append(ast.unparse(node))
    return "\n".join(imports) + "\n\n" + "\n\n".join(funcs)


def process_code(code: str, verbose: bool = False) -> str:
    # Limpieza de markdown fences y tokens especiales del chat template
    if "```python" in code:
        code = code.split("```python")[-1]
    code = code.replace("```", "").replace("<|im_end|>", "").replace("<|EOT|>", "")
    code = code.replace("`", "")  # los backticks sueltos rompen el parser

    # Intento 1: parsear tal cual
    try:
        return _extract_imports_and_funcs(ast.parse(code))
    except SyntaxError:
        pass

    # Intento 2: truncar líneas desde el final hasta que parsee
    lines = code.split("\n")
    total = len(lines)
    for i in range(total - 1, 0, -1):
        candidate = "\n".join(lines[:i])
        try:
            result = _extract_imports_and_funcs(ast.parse(candidate))
            n_trunc = total - i
            if verbose and n_trunc > 5:
                # Truncar demasiadas líneas suele indicar que el modelo
                # se cortó a mitad de función o devolvió mucha prosa.
                # Conviene revisar manualmente esos casos.
                print(f"  [process_code] truncadas {n_trunc}/{total} líneas finales")
            return result
        except SyntaxError:
            continue
    if verbose:
        print(f"  [process_code] no se pudo parsear nada — devolviendo raw")
    return code


# ── 3. ForbiddenTokensProcessor: bloquea keywords de Python ────────────────
import torch
from transformers import LogitsProcessor
PYTHON_KEYWORDS = [
    # OO no aplica en kernels Triton
    "class",
    # Construcciones funcionales/asíncronas que no van en kernels
    "lambda", "yield", "async", "await",
    # Manejo de excepciones no aplica dentro de @triton.jit
    "try", "except", "finally", "raise",
    # Context managers y scoping no aplican en kernels
    "with", "global", "nonlocal",
    # Aserciones y debug no aplican
    "assert", "del",
]


def _build_forbidden_token_ids(tokenizer, words):
    """Para cada keyword busca todas las variantes que el tokenizer codifica
    como UN solo token (con/sin espacio/newline/indent al frente).
    Solo masqueamos cuando es un único token, para no romper palabras legítimas
    que compartan prefijo (p.ej. 'asser' dentro de 'assertion')."""
    ids = set()
    for w in words:
        for variant in (w, " " + w, "\n" + w, "    " + w):
            toks = tokenizer.encode(variant, add_special_tokens=False)
            if len(toks) == 1:
                ids.add(toks[0])
    return sorted(ids)


def _sanity_check_forbidden(tokenizer, forbidden_ids):
    """Verifica que ningún token enmascarado decodifique a algo que la gramática
    necesita emitir. Si hay colisión, el modelo va a quedar atascado: la
    gramática exige un token y el processor lo bloquea — deadlock silencioso."""
    # Tokens que la gramática DEFINITIVAMENTE necesita poder emitir
    required = {"def", "return", "import", "from", "as", "if", "else", "for", "in", "and", "or"}
    conflicts = []
    for tok_id in forbidden_ids:
        decoded = tokenizer.decode([tok_id]).strip()
        if decoded in required:
            conflicts.append((tok_id, decoded))
    if conflicts:
        print(f"grammar↔processor: estos tokens están bloqueados")
        print(f"      pero la gramática los necesita:")
        for tok_id, decoded in conflicts:
            print(f"        id={tok_id}  decode={decoded!r}")
        print(f"      → revisar PYTHON_KEYWORDS y sacar los que correspondan")
    else:
        print(f"  ✓ sanity check: ningún token bloqueado conflictúa con la gramática")
    return conflicts


FORBIDDEN_IDS = _build_forbidden_token_ids(tokenizer, PYTHON_KEYWORDS)


class ForbiddenTokensProcessor(LogitsProcessor):
    """Pone -inf a una lista fija de token IDs en cada paso de decodificación.
    Funciona como defensa en profundidad junto a XGrammar: la gramática
    restringe la estructura sintáctica, este processor bloquea tokens que la
    gramática técnicamente permitiría dentro de `ident` pero que serían
    semánticamente inválidos (keywords de Python que no van en kernels)."""

    def __init__(self, forbidden_ids):
        self.forbidden = torch.tensor(forbidden_ids, dtype=torch.long)

    def __call__(self, input_ids, scores):
        scores[:, self.forbidden.to(scores.device)] = float("-inf")
        return scores


print(f"SYSTEM_PROMPT: {len(SYSTEM_PROMPT)} chars")
print(f"process_code: ready (usa verbose=True para diagnóstico de truncados)")
print(f"ForbiddenTokensProcessor: {len(FORBIDDEN_IDS)} keyword token ids masked")

# Sanity check: detectar colisiones grammar↔processor ANTES de generar.
# Si esto imprime conflictos, los reportes de exe@1 van a estar contaminados.
_sanity_check_forbidden(tokenizer, FORBIDDEN_IDS)

SYSTEM_PROMPT: 1424 chars
process_code: ready (usa verbose=True para diagnóstico de truncados)
ForbiddenTokensProcessor: 26 keyword token ids masked
  ✓ sanity check: ningún token bloqueado conflictúa con la gramática


[]

## 6. TritonBench-T setup

We use the standard TritonBench-T benchmark (166 PyTorch → Triton operators) and randomly sample 50 of them with a fixed seed for reproducibility.

The next cell clones TritonBench, installs the frontier SDKs (`anthropic`, `openai`), and patches a few hard-coded paths in the eval scripts so they work on a single-GPU Colab.


### 6.1 Clone TritonBench + install frontier SDKs

In [ ]:
# Instalar SDKs frontier y clonar/patchear TritonBench
import subprocess, os, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import anthropic
except ImportError:
    _pip("anthropic>=0.40.0")
    import anthropic

try:
    import openai
except ImportError:
    _pip("openai>=1.50.0")
    import openai

REPO_DIR = "/opt/TritonBench"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/thunlp/TritonBench.git", REPO_DIR],
                   check=True)
    print("TritonBench clonado")
else:
    print("TritonBench ya presente")

# Patches para Colab single-GPU
subprocess.run(["sed", "-i",
    "-e", f's|^statis_path = .*|statis_path = "{REPO_DIR}/data/TritonBench_T_v1.jsonl"|',
    "-e", f's|^py_folder = .*|py_folder = "{REPO_DIR}/data/TritonBench_T_v1/"|',
    "-e", 's|^py_interpreter = .*|import sys; py_interpreter = sys.executable|',
    f"{REPO_DIR}/EVAL/eval_T/0_call_acc.py"], check=True)
subprocess.run(["sed", "-i",
    "-e", f's|^gold_folder = .*|gold_folder = "{REPO_DIR}/data/TritonBench_T_v1/"|',
    "-e", 's|^py_interpreter = .*|import sys; py_interpreter = sys.executable|',
    f"{REPO_DIR}/EVAL/eval_T/1_exe_acc.py"], check=True)
subprocess.run(["sed", "-i", 's|^gpu_count = .*|gpu_count = 1|',
    f"{REPO_DIR}/performance_metrics/perf_T/run_bench/multiprocess_gpu_run.py"], check=True)

# Symlinks para los nombres call_acc / exe_acc (sin prefijo 0_/1_)
for src, dst in [("0_call_acc.py", "call_acc.py"), ("1_exe_acc.py", "exe_acc.py")]:
    dst_path = f"{REPO_DIR}/EVAL/eval_T/{dst}"
    if not os.path.exists(dst_path):
        os.symlink(f"{REPO_DIR}/EVAL/eval_T/{src}", dst_path)

print("TritonBench parcheado para single-GPU Colab")


TritonBench clonado
TritonBench parcheado para single-GPU Colab


### 6.2 Verify globals

Sanity-check that the model, grammar, helpers, and pattern library defined in the cells above are still live in this session.


In [ ]:
# Verifica que las definiciones de las secciones 2-5 están vivas en la sesión
_required = ["model", "tokenizer", "triton_grammar",
             "ForbiddenTokensProcessor", "FORBIDDEN_IDS",
             "process_code", "SYSTEM_PROMPT",
             "PATTERNS", "pick_pattern_for_instruction"]
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(
        f"Faltan globals: {_missing}\n"
        "Volvé a correr las celdas anteriores (§2-§5) en esta misma sesión."
    )

print(f"model:     {type(model).__name__}")
print(f"tokenizer: {tokenizer.__class__.__name__} (vocab={tokenizer.vocab_size})")
print(f"grammar:   compiled ({len(FORBIDDEN_IDS)} forbidden tokens)")
print(f"patterns:  {len(PATTERNS)} patterns + heuristic picker")
print("OK — globals listos")


model:     Qwen2ForCausalLM
tokenizer: Qwen2Tokenizer (vocab=151643)
grammar:   compiled (26 forbidden tokens)
patterns:  5 patterns + heuristic picker
OK — globals listos


### 6.3 API keys for frontier models

The frontier regimes (`claude`, `deepseek`) read API keys from Colab Secrets. If a key is missing, that regime is skipped silently.


In [ ]:
# Cargar API keys desde Colab Secrets si están disponibles
try:
    from google.colab import userdata
    for key in ("ANTHROPIC_API_KEY", "DEEPSEEK_API_KEY"):
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    pass

print("ANTHROPIC_API_KEY:", "OK" if os.environ.get("ANTHROPIC_API_KEY") else "MISSING (Claude regime se saltará)")
print("DEEPSEEK_API_KEY: ", "OK" if os.environ.get("DEEPSEEK_API_KEY")  else "MISSING (DeepSeek regime se saltará)")

# Inicializar clientes (lazy)
_anthropic_client = anthropic.Anthropic() if os.environ.get("ANTHROPIC_API_KEY") else None
_deepseek_client  = (
    openai.OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")
    if os.environ.get("DEEPSEEK_API_KEY") else None
)


ANTHROPIC_API_KEY: OK
DEEPSEEK_API_KEY:  OK


### 6.4 Select a subset of TritonBench-T operators

166 operators total. We sample `N_OPS` with `SEED=42` so the same 50 items are evaluated across regimes. Bump `N_OPS` to 166 for the full benchmark.


In [ ]:
import json, random
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────
N_OPS = 50          # 50 ≈ 2 h. Sube a 166 para benchmark completo (~5-6 h)
SEED  = 42          # Seed fija para reproducibilidad
# ───────────────────────────────────────────────────────────────────────────

all_items = json.loads(
    Path(f"{REPO_DIR}/data/TritonBench_T_simp_alpac_v1.json").read_text()
)
random.seed(SEED)
SUBSET = random.sample(all_items, min(N_OPS, len(all_items)))

print(f"Cargados {len(SUBSET)} operadores de TritonBench-T (de {len(all_items)} totales)")
print(f"Seed: {SEED}")
print()
print("Primeros 3 operadores:")
for i, item in enumerate(SUBSET[:3]):
    print(f"  [{i}] {item['instruction'][:80]}...")


Cargados 50 operadores de TritonBench-T (de 166 totales)
Seed: 42

Primeros 3 operadores:
  [0] You are an expert in Trion programming, capable of writing corresponding Triton ...
  [1] You are an expert in Trion programming, capable of writing corresponding Triton ...
  [2] You are an expert in Trion programming, capable of writing corresponding Triton ...


## 7. Generation for the 5 regimes

Three functions:

- `generate_qwen(item, regime)` — handles `qwen_free`, `qwen_constrained`, `qwen_patterns`. For `qwen_patterns` it picks a pattern with `pick_pattern_for_instruction(item["instruction"])` and injects it into the user prompt as a few-shot example.
- `generate_claude(item)` — unconstrained, Claude Sonnet 4.6 via the Anthropic SDK.
- `generate_deepseek(item)` — unconstrained, DeepSeek-V3 via the OpenAI-compatible SDK.

All three use the same `SYSTEM_PROMPT` so the only varying knobs are decoding strategy and the optional pattern.


In [ ]:
import time
import torch
from transformers import LogitsProcessorList
import xgrammar as xgr


def _build_qwen_user_msg(instruction: str, regime: str) -> str:
    """qwen_patterns inyecta un patrón few-shot; los otros usan la instrucción cruda."""
    if regime != "qwen_patterns":
        return instruction
    pattern_key = pick_pattern_for_instruction(instruction)
    example = PATTERNS[pattern_key]
    return (
        f"{instruction}\n\n"
        f"Reference Triton pattern (study the structure — the identifiers "
        f"`example_kernel`, `example_op`, `arg_tensor`, `scalar_factor`, etc. are "
        f"PLACEHOLDERS; adapt the wrapper signature to match the spec above and "
        f"do NOT copy these names verbatim):\n\n"
        f"```python\n{example}\n```\n"
    )


@torch.no_grad()
def generate_qwen(item, regime: str, max_new_tokens: int = 800) -> dict:
    """Genera código Triton para UN operador de TritonBench-T con el régimen indicado.
    Reusa model + tokenizer + grammar + ForbiddenTokens + PATTERNS de las celdas previas."""
    user_msg = _build_qwen_user_msg(item["instruction"], regime)
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)

    if regime == "qwen_free":
        logits_proc = None
    elif regime in ("qwen_constrained", "qwen_patterns"):
        logits_proc = LogitsProcessorList([
            xgr.contrib.hf.LogitsProcessor(triton_grammar),
            ForbiddenTokensProcessor(FORBIDDEN_IDS),
        ])
    else:
        raise ValueError(f"Régimen desconocido: {regime}")

    t0 = time.time()
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id,
                         logits_processor=logits_proc)
    dt = time.time() - t0
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return {"text": text, "time_s": dt, "n_new_tokens": out.shape[1] - ids["input_ids"].shape[1]}


def generate_claude(item, model_name: str = "claude-sonnet-4-6", max_tokens: int = 1024) -> dict:
    if _anthropic_client is None:
        return {"text": "[CLAUDE: no API key]", "time_s": 0.0, "n_new_tokens": 0}
    t0 = time.time()
    resp = _anthropic_client.messages.create(
        model=model_name, max_tokens=max_tokens, system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": item["instruction"]}],
    )
    dt = time.time() - t0
    return {"text": resp.content[0].text, "time_s": dt, "n_new_tokens": resp.usage.output_tokens}


def generate_deepseek(item, model_name: str = "deepseek-chat", max_tokens: int = 1024) -> dict:
    if _deepseek_client is None:
        return {"text": "[DEEPSEEK: no API key]", "time_s": 0.0, "n_new_tokens": 0}
    t0 = time.time()
    resp = _deepseek_client.chat.completions.create(
        model=model_name, max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": item["instruction"]},
        ],
    )
    dt = time.time() - t0
    return {"text": resp.choices[0].message.content, "time_s": dt, "n_new_tokens": resp.usage.completion_tokens}


print("Generadores listos: generate_qwen(item, regime), generate_claude(item), generate_deepseek(item)")
print(f"  qwen_patterns inyecta uno de {len(PATTERNS)} patrones según keywords del instruction")


Generadores listos: generate_qwen(item, regime), generate_claude(item), generate_deepseek(item)
  qwen_patterns inyecta uno de 5 patrones según keywords del instruction


## 8. Run generation for all 5 regimes

Qwen regimes are GPU-bound and sequential (~30–60 s/op). Frontier regimes are API-bound and parallel (~5–15 s/op).


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import json

REGIMES = ["qwen_free", "qwen_constrained", "qwen_patterns", "claude", "deepseek"]

# Persistencia local. Si Colab muere a mitad de la corrida, podés re-correr esta
# celda y arranca desde el último idx guardado (resume automático).
PRED_DIR = Path("./tb_predictions")
PRED_DIR.mkdir(exist_ok=True)
print(f"Predictions en: {PRED_DIR.resolve()}")

# ── 2. Resume: cargar lo que ya estaba guardado ─────────────────────────
def load_existing(regime):
    path = PRED_DIR / f"predictions_{regime}.jsonl"
    if not path.exists():
        return []
    out = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

all_predictions = {r: load_existing(r) for r in REGIMES}
print("\nEstado actual:")
for r, p in all_predictions.items():
    print(f"  {r}: {len(p)}/{len(SUBSET)}")

# ── 3. Append atómico: guarda UNA predicción al .jsonl del régimen ─────
def save_prediction(regime, idx, instruction, predict):
    path = PRED_DIR / f"predictions_{regime}.jsonl"
    rec = {"idx": idx, "instruction": instruction, "predict": predict}
    with path.open("a") as f:
        f.write(json.dumps(rec) + "\n")
        f.flush()

# ── 4. Qué índices faltan por régimen ───────────────────────────────────
def missing_indices(regime):
    done = {p.get("idx") for p in all_predictions[regime] if "idx" in p}
    return [i for i in range(len(SUBSET)) if i not in done]

total_qwen_jobs  = sum(len(missing_indices(r)) for r in ("qwen_free", "qwen_constrained", "qwen_patterns"))
total_front_jobs = sum(len(missing_indices(r)) for r in ("claude", "deepseek"))
est_min = (total_qwen_jobs * 45 + total_front_jobs * 10) / 60
print(f"\nFaltan: {total_qwen_jobs} Qwen + {total_front_jobs} frontier. Estimación: ~{est_min:.0f} min")

# ── 5. Qwen regimes (secuenciales, GPU-bound) ───────────────────────────
for regime in ("qwen_free", "qwen_constrained", "qwen_patterns"):
    missing = missing_indices(regime)
    if not missing:
        print(f"\n=== {regime}: ya completo, skip ===")
        continue
    print(f"\n=== {regime} ({len(missing)} ops faltantes) ===")
    for i in missing:
        item = SUBSET[i]
        try:
            res = generate_qwen(item, regime)
            code_out = process_code(res["text"])
        except Exception as e:
            code_out = f"# generation failed: {e}\n"
        save_prediction(regime, i, item["instruction"], code_out)
        all_predictions[regime].append({"idx": i, "instruction": item["instruction"], "predict": code_out})
        n_done = len(all_predictions[regime])
        if n_done % 5 == 0 or i == missing[-1]:
            print(f"  [{regime}] {n_done}/{len(SUBSET)}")

# ── 6. Frontier regimes (paralelos, API-bound) ──────────────────────────
def _frontier_one(args):
    regime, idx, item = args
    fn = generate_claude if regime == "claude" else generate_deepseek
    try:
        res = fn(item)
        code_out = process_code(res["text"])
    except Exception as e:
        code_out = f"# generation failed: {e}\n"
    return regime, idx, item["instruction"], code_out

frontier_jobs = []
for regime in ("claude", "deepseek"):
    available = (_anthropic_client is not None) if regime == "claude" else (_deepseek_client is not None)
    if not available:
        print(f"\n=== {regime}: SKIPPED (no API key) ===")
        continue
    missing = missing_indices(regime)
    if not missing:
        print(f"\n=== {regime}: ya completo, skip ===")
        continue
    print(f"\n=== {regime} ({len(missing)} ops faltantes, paralelo) ===")
    for i in missing:
        frontier_jobs.append((regime, i, SUBSET[i]))

if frontier_jobs:
    with ThreadPoolExecutor(max_workers=4) as ex:
        futs = [ex.submit(_frontier_one, job) for job in frontier_jobs]
        done = 0
        for fut in as_completed(futs):
            regime, idx, instruction, predict = fut.result()
            save_prediction(regime, idx, instruction, predict)
            all_predictions[regime].append({"idx": idx, "instruction": instruction, "predict": predict})
            done += 1
            if done % 10 == 0 or done == len(frontier_jobs):
                print(f"  [frontier] {done}/{len(frontier_jobs)}")

# ── 7. Normalizar: ordenar por idx + quitar campo `idx` (formato TritonBench) ──
for regime in REGIMES:
    path = PRED_DIR / f"predictions_{regime}.jsonl"
    if not path.exists():
        continue
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    by_idx = {}
    for r in records:
        if "idx" in r:
            by_idx[r["idx"]] = r
    sorted_records = [by_idx[i] for i in sorted(by_idx)]
    with path.open("w") as f:
        for r in sorted_records:
            f.write(json.dumps({"instruction": r["instruction"], "predict": r["predict"]}) + "\n")
    print(f"  Normalizado {regime}: {len(sorted_records)} predictions")

print(f"\n✓ Listo. Archivos finales en {PRED_DIR.resolve()}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Predictions en: /content/tb_predictions

Estado actual:
  qwen_free: 0/50
  qwen_constrained: 0/50
  qwen_patterns: 0/50
  claude: 0/50
  deepseek: 0/50

Faltan: 150 Qwen + 100 frontier. Estimación: ~129 min

=== qwen_free (50 ops faltantes) ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [qwen_free] 5/50
  [qwen_free] 10/50
  [qwen_free] 15/50
  [qwen_free] 20/50
  [qwen_free] 25/50
  [qwen_free] 30/50
  [qwen_free] 35/50
  [qwen_free] 40/50
  [qwen_free] 45/50
  [qwen_free] 50/50

=== qwen_constrained (50 ops faltantes) ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [qwen_constrained] 5/50
  [qwen_constrained] 10/50
  [qwen_constrained] 15/50
  [qwen_constrained] 20/50
  [qwen_constrained] 25/50
  [qwen_constrained] 30/50
  [qwen_constrained] 35/50
  [qwen_constrained] 40/50
  [qwen_constrained] 45/50
  [qwen_constrained] 50/50

=== qwen_patterns (50 ops faltantes) ===
  [qwen_patterns] 5/50
  [qwen_patterns] 10/50
  [qwen_patterns] 15/50
  [qwen_patterns] 20/50
  [qwen_patterns] 25/50
  [qwen_patterns] 30/50
  [qwen_patterns] 35/50
  [qwen_patterns] 40/50
  [qwen_patterns] 45/50
  [qwen_patterns] 50/50

=== claude (50 ops faltantes, paralelo) ===

=== deepseek (50 ops faltantes, paralelo) ===
  [frontier] 10/100
  [frontier] 20/100
  [frontier] 30/100
  [frontier] 40/100
  [frontier] 50/100
  [frontier] 60/100
  [frontier] 70/100
  [frontier] 80/100
  [frontier] 90/100
  [frontier] 100/100
  Normalizado qwen_free: 50 predictions
  Normalizado qwen_constrained: 50 predictions
  Normalizado qwen_patterns: 50 predictions
  Normalizado claude: 50 p

## 9. TritonBench-T evaluation — Phase 1 (`call@1`) + Phase 2 (`exe@1`)

Phase 1 checks each kernel **runs without crashing**. Phase 2 checks the output **matches the PyTorch reference** numerically. These are the same metrics as the literature reports.


In [ ]:
import shutil

if f"{REPO_DIR}/EVAL/eval_T" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/EVAL/eval_T")
os.environ["PYTHONPATH"] = f"{REPO_DIR}/EVAL/eval_T" + os.pathsep + os.environ.get("PYTHONPATH", "")

import call_acc as _call_acc_mod
import exe_acc  as _exe_acc_mod


def run_tritonbench_eval(predictions_path: Path, label: str) -> dict:
    """Corre Phase 1 + Phase 2 de TritonBench sobre un archivo de predictions."""
    out_dir = Path(f"/content/tb_results_{label}")
    if out_dir.exists():
        shutil.rmtree(out_dir)
    call_dir = out_dir / "call_acc"
    call_dir.mkdir(parents=True)

    # Phase 1
    print(f"\n[{label}] Phase 1: call_acc...")
    _call_acc_mod.call_4file(str(predictions_path), str(call_dir), gpus=[0])
    survivors_1 = sorted(p.name for p in call_dir.glob("*.py"))
    total = sum(1 for _ in open(predictions_path))
    print(f"[{label}] Phase 1: {len(survivors_1)}/{total} ({100*len(survivors_1)/total:.1f}%)")

    # Phase 2
    if survivors_1:
        print(f"[{label}] Phase 2: exe_acc...")
        _exe_acc_mod.execute_4folder(str(call_dir), gpus=[0])
        survivors_2 = sorted(p.name for p in call_dir.glob("*.py"))
        print(f"[{label}] Phase 2: {len(survivors_2)}/{total} ({100*len(survivors_2)/total:.1f}%)")
    else:
        survivors_2 = []

    return {
        "total":    total,
        "call_acc": len(survivors_1),
        "exe_acc":  len(survivors_2),
        "call_dir": str(call_dir),
    }


# Correr eval para cada régimen
results = {}
for regime in REGIMES:
    pred_path = PRED_DIR / f"predictions_{regime}.jsonl"
    if not pred_path.exists():
        print(f"Skip {regime}: no predictions file")
        continue
    print("\n" + "=" * 80)
    print(f"  EVALUATING REGIME: {regime}")
    print("=" * 80)
    try:
        results[regime] = run_tritonbench_eval(pred_path, regime)
    except Exception as e:
        print(f"WARN {regime} eval crashed: {type(e).__name__}: {e}")
        results[regime] = {"total": 0, "call_acc": 0, "exe_acc": 0, "call_dir": None, "error": str(e)}



  EVALUATING REGIME: qwen_free

[qwen_free] Phase 1: call_acc...
instruction
=== Output for cos_signbit.py on GPU 0 ===

=== Errors for cos_signbit.py on GPU 0 ===
Traceback (most recent call last):
  File "/content/tb_results_qwen_free/call_acc/cos_signbit.py", line 16, in <module>
    def cos_signbit(input: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
                                            ^^^^^
NameError: name 'Tuple' is not defined. Did you mean: 'tuple'?

=== Output for silu_batch_norm.py on GPU 0 ===

=== Errors for silu_batch_norm.py on GPU 0 ===
Traceback (most recent call last):
  File "/content/tb_results_qwen_free/call_acc/silu_batch_norm.py", line 64, in <module>
    test_results = test_silu_batch_norm()
                   ^^^^^^^^^^^^^^^^^^^^^^
  File "/content/tb_results_qwen_free/call_acc/silu_batch_norm.py", line 49, in test_silu_batch_norm
    results["test_case_1"] = silu_batch_norm(input_tensor, running_mean, running_var, training=False)
                

## 10. Phase 3 — Efficiency (speedup vs PyTorch)

Phase 3 benchmarks every surviving kernel against a PyTorch reference (`triton.testing.do_bench`). It is the most expensive phase: **~30–60 min per regime**. Set `RUN_PHASE_3 = False` to skip.

We only run Phase 3 on regimes with at least `PHASE3_MIN_SURVIVORS` exe_acc passes — there's no point benchmarking 0 or 1 kernels.


In [ ]:
RUN_PHASE_3 = True          # False para saltar (~30-60 min por régimen)
PHASE3_MIN_SURVIVORS = 5    # umbral mínimo de exe_acc para correr efficiency

if RUN_PHASE_3:
    perf_root = f"{REPO_DIR}/performance_metrics/perf_T"
    for regime, r in results.items():
        if r["exe_acc"] < PHASE3_MIN_SURVIVORS or r.get("call_dir") is None:
            print(f"Skip Phase 3 [{regime}]: solo {r['exe_acc']} survivors (mín {PHASE3_MIN_SURVIVORS})")
            r["speedup"] = None
            continue

        print(f"\n=== Phase 3: {regime} ===")
        perf_results_dir = Path(f"/content/tb_results_{regime}/perf_results")
        perf_results_dir.mkdir(parents=True, exist_ok=True)

        # 3a: escribir per-op benchmark scripts
        print(f"  [{regime}] 3a: writing bench scripts")
        subprocess.run([sys.executable, "run_bench/write_file.py",
                        "--input_folder_path", r["call_dir"],
                        "--results_path",      str(perf_results_dir)],
                       cwd=perf_root, check=True)

        # 3b: correr benchmarks (GPU-bound, lento)
        print(f"  [{regime}] 3b: running benchmarks")
        subprocess.run([sys.executable, "run_bench/multiprocess_gpu_run.py"],
                       cwd=perf_root, check=True)

        # 3c: computar speedup final
        print(f"  [{regime}] 3c: computing speedup")
        eff = subprocess.run([sys.executable, "2_efficiency.py",
                              "--gen_folder", str(perf_results_dir)],
                             cwd=f"{REPO_DIR}/EVAL/eval_T",
                             capture_output=True, text=True)
        speedup = None
        for line in eff.stdout.splitlines():
            if line.startswith("speed up:"):
                try:
                    speedup = float(line.split(":", 1)[1].strip())
                except ValueError:
                    pass
        r["speedup"] = speedup
        print(f"[{regime}] Phase 3: speedup = {speedup}")
else:
    print("Phase 3 saltado (RUN_PHASE_3 = False)")
    for r in results.values():
        r["speedup"] = None


Skip Phase 3 [qwen_free]: solo 2 survivors (mín 5)

=== Phase 3: qwen_constrained ===
  [qwen_constrained] 3a: writing bench scripts
  [qwen_constrained] 3b: running benchmarks
  [qwen_constrained] 3c: computing speedup
[qwen_constrained] Phase 3: speedup = 5.3

=== Phase 3: qwen_patterns ===
  [qwen_patterns] 3a: writing bench scripts
  [qwen_patterns] 3b: running benchmarks
  [qwen_patterns] 3c: computing speedup
[qwen_patterns] Phase 3: speedup = 7.58

=== Phase 3: claude ===
  [claude] 3a: writing bench scripts
  [claude] 3b: running benchmarks
  [claude] 3c: computing speedup
[claude] Phase 3: speedup = 1.66

=== Phase 3: deepseek ===
  [deepseek] 3a: writing bench scripts
  [deepseek] 3b: running benchmarks
  [deepseek] 3c: computing speedup
[deepseek] Phase 3: speedup = 1.16


## 11. Final comparison table

Side-by-side numbers, directly comparable to published TritonBench-T results.


In [ ]:
import pandas as pd

rows = []
for regime in REGIMES:
    r = results.get(regime, {})
    total = r.get("total", 0)
    ca    = r.get("call_acc", 0)
    ea    = r.get("exe_acc", 0)
    sp    = r.get("speedup")
    rows.append({
        "regime":      regime,
        "n_ops":       total,
        "call_acc":    f"{ca}/{total} ({100*ca/total:.1f}%)" if total else "—",
        "exe_acc":     f"{ea}/{total} ({100*ea/total:.1f}%)" if total else "—",
        "speedup":     f"{sp:.2f}x" if sp else "—",
    })

df = pd.DataFrame(rows)
print("=" * 90)
print("  TRITONBENCH-T RESULTS — 5 regimes side-by-side")
print("=" * 90)
print(df.to_string(index=False))
print()
print("Reference (Claude API on full 166 ops, prior run): "
      "60% call@1, 60% exe@1, 1.09x speedup")

# Mostrar también el dataframe para que Jupyter lo renderice bonito
df


  TRITONBENCH-T RESULTS — 5 regimes side-by-side
          regime  n_ops      call_acc       exe_acc speedup
       qwen_free     50   2/50 (4.0%)   2/50 (4.0%)       —
qwen_constrained     50 15/50 (30.0%) 15/50 (30.0%)   5.30x
   qwen_patterns     50 15/50 (30.0%) 15/50 (30.0%)   7.58x
          claude     50 34/50 (68.0%) 34/50 (68.0%)   1.66x
        deepseek     50 28/50 (56.0%) 28/50 (56.0%)   1.16x

Reference (Claude API on full 166 ops, prior run): 60% call@1, 60% exe@1, 1.09x speedup


,regime,n_ops,call_acc,exe_acc,speedup
0,qwen_free,50,2/50 (4.0%),2/50 (4.0%),—
1,qwen_constrained,50,15/50 (30.0%),15/50 (30.0%),5.30x
2,qwen_patterns,50,15/50 (30.0%),15/50 (30.0%),7.58x
3,claude,50,34/50 (68.0%),34/50 (68.0%),1.66x
4,deepseek,50,28/50 (56.0%),28/50 (56.0%),1.16x


## 12. Save backup (download from Colab)


In [ ]:
# Backup de todos los resultados (descarga si Colab muere)
try:
    from google.colab import files
    shutil.make_archive("/content/tb_full_backup", "zip", "/content")
    print("Backup creado en /content/tb_full_backup.zip")
    print("Para descargar:  files.download('/content/tb_full_backup.zip')")
except ImportError:
    print("Fuera de Colab — backup saltado")

# Guardar también el JSON de resultados (compacto)
results_summary = {regime: {k: v for k, v in r.items() if k != "call_dir"}
                   for regime, r in results.items()}
Path("/content/results_summary.json").write_text(json.dumps(results_summary, indent=2))
print("Resumen JSON en /content/results_summary.json")


Backup creado en /content/tb_full_backup.zip
Para descargar:  files.download('/content/tb_full_backup.zip')
Resumen JSON en /content/results_summary.json


## 13. Discussion

This table answers the research question: **do XGrammar-constrained decoding and a few-shot pattern library improve `call@1` / `exe@1` for a small open-source coder model on TritonBench-T**, and how does the small model compare against two frontier APIs?

### How to read the table

1. **Does grammar help Qwen?** Compare `qwen_free` vs `qwen_constrained`. A gap on `call@1` confirms that masking invalid tokens at decode time prevents the most common failure mode (garbled output, missing imports, wrong skeleton).

2. **Do patterns add anything beyond grammar?** Compare `qwen_constrained` vs `qwen_patterns`. A positive gap means the few-shot pattern (picked by `pick_pattern_for_instruction`) is helping the model land on the right Triton idiom for the operation. A *negative* gap means the pattern is pulling the model away from the correct solution — worth inspecting which operators regressed.

3. **How much of the frontier gap does our method close?** Compute `(qwen_patterns − qwen_free) / (claude − qwen_free)`. >50% would be a strong methodological result for a 7B local model.

4. **Is `exe@1` bottlenecked elsewhere?** If `call@1` rises but `exe@1` stays flat, the remaining failures are *semantic* (wrong math, wrong masking, wrong reduction axis) — not something XGrammar or patterns can fix without richer guidance.

5. **Speedup as a tie-breaker.** Among the kernels that pass `exe@1`, do all regimes produce equally fast Triton, or is one model writing measurably more efficient code?

### Limitations

- **50/166 ops with a fixed seed.** Bump `N_OPS` to 166 in §6.4 for the full benchmark; the table format is unchanged.
- **Single-shot, greedy decoding.** No `pass@k`, no temperature sweep — the contribution we're measuring is the *decoding constraint*, not sampling tricks.
- **One pattern library, one heuristic.** `pick_pattern_for_instruction` is a keyword matcher over 5 patterns. Failure analysis on `qwen_patterns` vs `qwen_constrained` will tell you whether the bottleneck is pattern coverage, picker accuracy, or something the grammar already handles.
